# LeafLens — Training Notebook (Google Colab)

This notebook reproduces the crop-disease classifier on a free Colab GPU. It clones
the project repo and runs the exact same `prepare_data.py` and `train.py` scripts used
locally, so the results match what the deployed backend serves.

**Datasets (public, no login needed):**
- `BrandonFors/Plant-Diseases-PlantVillage-Dataset` — potato, tomato, corn, pepper
- `sharmin3/Rice-Leaf-Disease` — rice

**Model:** MobileNetV2 (ImageNet-pretrained) with a fine-tuned classifier head.

Runtime -> Change runtime type -> T4 GPU is recommended (CPU also works, just slower).

In [ ]:
# 1. Get the code and install dependencies
!git clone https://github.com/ShadmanSakibRahman/leaflens.git
%cd leaflens
!pip install -q -r ml/requirements.txt

In [ ]:
# 2. Download the datasets and build a clean ImageFolder with our canonical classes
!python ml/prepare_data.py --cap 800

In [ ]:
# 3. Train (GPU is auto-detected). Writes weights/labels/metrics into backend/model/
!python ml/train.py --epochs 12 --per-class-cap 800

In [ ]:
# 4. Look at the results
import json
with open('backend/model/metrics.json') as f:
    m = json.load(f)
print('Test accuracy :', m['test_accuracy'])
print('Macro F1      :', m['macro_f1'])
print('Classes       :', m['num_classes'])

from IPython.display import Image
Image('backend/model/confusion_matrix.png')

In [ ]:
# 5. Download the trained artifacts to plug into the backend
!cd backend/model && zip -r /content/model_artifacts.zip weights.pt labels.json metrics.json confusion_matrix.png
from google.colab import files
files.download('/content/model_artifacts.zip')

## Notes

- The backbone (MobileNetV2 features) is frozen and only the classifier head is trained.
  This is fast and enough because ImageNet features already capture leaf texture well.
- `prepare_data.py` caps images per class so training stays quick and balanced.
- Everything the API needs is in `backend/model/`: `weights.pt` and `labels.json`.

Author: Md. Shadman Sakib Rahman